# Internship Aggregator — Colab interface

```
 Colab (this notebook)            PostgreSQL (Neon)            GitHub Actions
 web UI: register, search   ──►   internships / users / sent  ◄──  every 6 h: Scrapy scrapes,
 and subscribe                    (one shared database)             then sends the e-mail digests
```

This notebook **only serves the web UI**. It does not scrape and does not send e-mails:
GitHub Actions does that (with **Scrapy**, branch `scrapy-scraping`) and writes into the same database
that the interface reads.

**Colab secrets needed** (🔑 panel, switch *Notebook access* on for each):

| Secret | Why |
|---|---|
| `DATABASE_URL` | the shared Neon/PostgreSQL database |
| `SECRET_KEY` | **must be the same value as the GitHub Actions secret**, otherwise the unsubscribe links in the e-mails are rejected by this UI |
| `NGROK_TOKEN` | public URL for the interface |
| `GITHUB_TOKEN` *(optional)* | only if the repository is private |

Run the cells top to bottom.

**Cell 1 — Install** (UI dependencies only; no scraping libraries needed here)

In [ ]:
!pip -q install fastapi "uvicorn[standard]" python-multipart "bcrypt==4.1.3" \
                itsdangerous psycopg2-binary pyngrok
print("Install done.")

**Cell 2 — Get the code from GitHub** (single source of truth: the notebook imports `web.py`, it does not copy it)

In [ ]:
import os, subprocess, sys
from google.colab import userdata

REPO   = "WardiSafae/internship-digest"
BRANCH = "scrapy-scraping"          # use "main" for the requests + BeautifulSoup version
DEST   = "/content/internship-digest"

def secret(name, default=""):
    try:
        v = userdata.get(name)
        return v if v else default
    except Exception:
        return os.environ.get(name, default)

token = secret("GITHUB_TOKEN")
url = (f"https://{token}@github.com/{REPO}.git" if token
       else f"https://github.com/{REPO}.git")

if os.path.isdir(DEST):
    subprocess.run(["git", "-C", DEST, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, DEST], check=True)

if DEST not in sys.path:
    sys.path.insert(0, DEST)
os.chdir(DEST)
print("Code ready on branch:", subprocess.check_output(
    ["git", "-C", DEST, "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())

**Cell 3 — Configuration from Colab secrets**

In [ ]:
os.environ["DATABASE_URL"] = secret("DATABASE_URL")
os.environ["SECRET_KEY"]   = secret("SECRET_KEY")
os.environ["NGROK_TOKEN"]  = secret("NGROK_TOKEN")
os.environ["DRY_RUN"]      = "1"      # the UI must never send e-mail; GitHub Actions does that

missing = [k for k in ("DATABASE_URL", "SECRET_KEY", "NGROK_TOKEN") if not os.environ[k]]
for k in ("DATABASE_URL", "SECRET_KEY", "NGROK_TOKEN"):
    print(("✅" if os.environ[k] else "❌"), k)
if missing:
    raise RuntimeError(f"Add these secrets in the 🔑 panel first: {missing}")

**Cell 4 — Check the shared database** (what GitHub Actions has produced so far)

In [ ]:
import psycopg2
con = psycopg2.connect(os.environ["DATABASE_URL"], connect_timeout=10)
cur = con.cursor()
cur.execute("""SELECT source, COUNT(*), MAX(scraped_at)
               FROM internships GROUP BY source ORDER BY 2 DESC""")
rows = cur.fetchall()
print(f"{'source':18s} {'offers':>6s}   last scraped (UTC)")
for source, n, last in rows:
    print(f"{source:18s} {n:6d}   {(last or '')[:19]}")
cur.execute("SELECT COUNT(*) FROM internships"); print("\nTotal offers:", cur.fetchone()[0])
cur.execute("SELECT COUNT(*) FROM users WHERE subscribed=1"); print("Subscribers :", cur.fetchone()[0])
con.close()

**Cell 5 — Launch the interface (uvicorn + ngrok)** — re-running this cell restarts the server cleanly

In [ ]:
import socket, threading, time
import uvicorn
from pyngrok import ngrok, conf

PORT = 8000

# stop a previous run of this cell, if any
if "server" in globals():
    server.should_exit = True
    SERVER_THREAD.join(timeout=10)
try:
    ngrok.kill()
except Exception:
    pass

# import AFTER the environment is configured (db.py reads DATABASE_URL at import)
import importlib, web
importlib.reload(web)

conf.get_default().auth_token = os.environ["NGROK_TOKEN"]
config = uvicorn.Config(web.app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)
SERVER_THREAD = threading.Thread(target=server.run, name="uvicorn", daemon=True)
SERVER_THREAD.start()

for _ in range(60):                       # wait until the port answers
    try:
        socket.create_connection(("127.0.0.1", PORT), timeout=1).close()
        break
    except OSError:
        time.sleep(0.25)
else:
    raise RuntimeError("uvicorn did not start")

public_url = ngrok.connect(PORT, bind_tls=True).public_url
os.environ["APP_BASE_URL"] = public_url
print("🌐 Interface:", public_url)
print("   Register → set keywords → Subscribe. The e-mails come from GitHub Actions.")

**Optional — stop the interface**

In [ ]:
server.should_exit = True
ngrok.kill()
print("Stopped.")